# 03 Evaluate 2D CNN v0.2

**Notebook version:** v0.2  
**Category:** evaluation  
**Purpose:** Select a trained model artifact via `ipywidgets`, then evaluate it on `./validation-data` with regression diagnostics.  
**Inputs:** `./models/*_2d-cnn*/`, `./validation-data`, `./src/evaluate.py`  
**Expected outputs:** `evaluation_summary.json`, prediction plots, and tabular sample predictions in selected run directory.  
**Artifact write mode:** canonical run-linked evaluation artifacts written  
**Decision supported:** `READY_FOR_NEXT_DECISION` vs `PATCH_REQUIRED`


In [4]:
# Repo Setup
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    for parent in repo_root.parents:
        if (parent / 'src').exists() and (parent / 'models').exists():
            repo_root = parent
            break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'repo_root={repo_root}')


repo_root=/home/mitch/development/raccoon-ball/rb-training


In [5]:
# Eval Config + Selector
import ipywidgets as widgets
from datetime import datetime

MODELS_ROOT = (repo_root / 'models').resolve()
if not MODELS_ROOT.exists():
    raise FileNotFoundError(f'Models root missing: {MODELS_ROOT}')

EVAL_READY = False
EVAL_CONFIG_TO_RUN = None
EVAL_START_REQUESTED_AT = None


def _dedupe_paths(paths):
    seen = set()
    deduped = []
    for path in paths:
        resolved = path.resolve()
        key = str(resolved)
        if key not in seen:
            deduped.append(resolved)
            seen.add(key)
    return deduped


def _discover_model_dirs(models_root):
    model_dirs = [path.resolve() for path in models_root.glob('*_2d-cnn*') if path.is_dir()]
    return sorted(model_dirs, key=lambda path: path.name)


def _discover_run_dirs(model_dir):
    candidates = []
    if (model_dir / 'config.json').exists():
        candidates.append(model_dir)

    for base in (model_dir, model_dir / 'runs'):
        if not base.exists() or not base.is_dir():
            continue
        for child in sorted(base.glob('run_*')):
            if child.is_dir():
                candidates.append(child.resolve())

    run_register_path = model_dir / 'run_register.json'
    if run_register_path.exists():
        with run_register_path.open('r', encoding='utf-8') as handle:
            payload = json.load(handle)
        runs = payload.get('runs', [])
        if isinstance(runs, list):
            for row in runs:
                if not isinstance(row, dict):
                    continue
                raw = row.get('run_dir')
                if not raw:
                    continue
                run_path = Path(str(raw))
                if not run_path.is_absolute():
                    run_path = (repo_root / run_path).resolve()
                if run_path.is_dir():
                    candidates.append(run_path)

    valid = [path for path in _dedupe_paths(candidates) if (path / 'config.json').exists()]
    return sorted(valid, key=lambda path: path.name)


def _to_repo_rel(path):
    try:
        return str(path.resolve().relative_to(repo_root))
    except ValueError:
        return str(path.resolve())


model_dirs = _discover_model_dirs(MODELS_ROOT)
if not model_dirs:
    raise FileNotFoundError(f'No model directories found under {MODELS_ROOT}')

model_dropdown = widgets.Dropdown(
    options=[(path.name, str(path)) for path in model_dirs],
    value=str(model_dirs[-1]),
    description='Model',
    layout=widgets.Layout(width='600px'),
)
run_dropdown = widgets.Dropdown(
    options=[],
    description='Run',
    layout=widgets.Layout(width='600px'),
)
batch_size_input = widgets.BoundedIntText(value=4, min=1, max=2048, description='Batch')
evaluate_internal_test_checkbox = widgets.Checkbox(
    value=False,
    description='Eval test_internal if present',
)
start_eval_button = widgets.Button(
    description='Start Validation',
    button_style='success',
    icon='play',
    tooltip='Click to arm evaluation for the selected run',
)
status_html = widgets.HTML(value='')
launch_html = widgets.HTML(value='<i>Select model/run, then click Start Validation.</i>')


def _set_not_ready(message):
    global EVAL_READY, EVAL_CONFIG_TO_RUN, EVAL_START_REQUESTED_AT
    EVAL_READY = False
    EVAL_CONFIG_TO_RUN = None
    EVAL_START_REQUESTED_AT = None
    launch_html.value = f'<i>{message}</i>'


def _apply_eval_config(*_):
    global MODEL_RUN_DIRECTORY, EVAL_CONFIG
    MODEL_RUN_DIRECTORY = Path(run_dropdown.value).resolve()
    EVAL_CONFIG = {
        'model_run_directory': str(MODEL_RUN_DIRECTORY),
        'validation_data_root': str(repo_root / 'validation-data'),
        'training_data_root': str(repo_root / 'training-data'),
        'batch_size': int(batch_size_input.value),
        'evaluate_internal_test_if_present': bool(evaluate_internal_test_checkbox.value),
        'entrypoint_type': 'notebook',
        'entrypoint_path': 'notebooks/03_evaluate_2d_cnn_v0.2.ipynb',
    }
    status_html.value = (
        '<b>Selection ready.</b><br>'
        f"model: {_to_repo_rel(Path(model_dropdown.value))}<br>"
        f"run: {_to_repo_rel(MODEL_RUN_DIRECTORY)}<br>"
        f"batch_size: {int(batch_size_input.value)}<br>"
        f"evaluate_internal_test_if_present: {bool(evaluate_internal_test_checkbox.value)}"
    )
    _set_not_ready('Selection changed. Click <b>Start Validation</b> to continue to Eval Execute.')


def _refresh_runs(*_):
    selected_model = Path(model_dropdown.value).resolve()
    run_dirs = _discover_run_dirs(selected_model)
    if not run_dirs:
        raise FileNotFoundError(
            f'No runnable run directories with config.json found under {selected_model}'
        )
    run_dropdown.options = [(_to_repo_rel(path), str(path)) for path in run_dirs]
    run_dropdown.value = str(run_dirs[-1])
    _apply_eval_config()


def _on_start_eval_clicked(_):
    global EVAL_READY, EVAL_CONFIG_TO_RUN, EVAL_START_REQUESTED_AT
    if 'EVAL_CONFIG' not in globals() or not EVAL_CONFIG:
        raise RuntimeError('EVAL_CONFIG not available. Re-run this cell.')
    EVAL_CONFIG_TO_RUN = dict(EVAL_CONFIG)
    EVAL_START_REQUESTED_AT = datetime.now().isoformat(timespec='seconds')
    EVAL_READY = True
    launch_html.value = (
        '<b>Validation armed.</b><br>'
        f"requested_at: {EVAL_START_REQUESTED_AT}<br>"
        'Now run the <b># Eval Execute</b> cell.'
    )
    print('Validation armed for', EVAL_CONFIG_TO_RUN['model_run_directory'])


model_dropdown.observe(_refresh_runs, names='value')
run_dropdown.observe(_apply_eval_config, names='value')
batch_size_input.observe(_apply_eval_config, names='value')
evaluate_internal_test_checkbox.observe(_apply_eval_config, names='value')
start_eval_button.on_click(_on_start_eval_clicked)

ui = widgets.VBox([
    widgets.HTML('<b>Select Model Artifact To Evaluate</b>'),
    model_dropdown,
    run_dropdown,
    widgets.HBox([batch_size_input, evaluate_internal_test_checkbox]),
    start_eval_button,
    status_html,
    launch_html,
])
display(ui)

_refresh_runs()
print('MODEL_RUN_DIRECTORY', MODEL_RUN_DIRECTORY)
print('Click "Start Validation" when ready to run evaluation.')
EVAL_CONFIG


MODEL_RUN_DIRECTORY /home/mitch/development/raccoon-ball/rb-training/models/260331-2016_2d-cnn/runs/run_0001
Click "Start Validation" when ready to run evaluation.


{'model_run_directory': '/home/mitch/development/raccoon-ball/rb-training/models/260331-2016_2d-cnn/runs/run_0001',
 'validation_data_root': '/home/mitch/development/raccoon-ball/rb-training/validation-data',
 'training_data_root': '/home/mitch/development/raccoon-ball/rb-training/training-data',
 'batch_size': 4,
 'evaluate_internal_test_if_present': False,
 'entrypoint_type': 'notebook',
 'entrypoint_path': 'notebooks/03_evaluate_2d_cnn_v0.2.ipynb'}

In [6]:
# Eval Execute
from src.evaluate import evaluate_saved_run

if 'EVAL_CONFIG' not in globals():
    raise RuntimeError('EVAL_CONFIG not found. Run the Eval Config + Selector cell first.')
if not bool(globals().get('EVAL_READY', False)):
    raise RuntimeError(
        'Validation not started. In the selector cell, choose model/run and click "Start Validation", then rerun this cell.'
    )

config_to_run = dict(globals().get('EVAL_CONFIG_TO_RUN') or EVAL_CONFIG)
eval_result = evaluate_saved_run(config_to_run)
RUN_DIR = Path(eval_result['summary']['run_dir'])
print('RUN_DIR', RUN_DIR)

globals()['EVAL_READY'] = False
globals()['EVAL_CONFIG_TO_RUN'] = None
print('Validation completed. Click "Start Validation" again before the next eval run.')

eval_result['summary']


RuntimeError: Validation not started. In the selector cell, choose model/run and click "Start Validation", then rerun this cell.

In [ ]:
# Metrics View
with (RUN_DIR / 'evaluation_summary.json').open('r', encoding='utf-8') as handle:
    evaluation_summary = json.load(handle)

print('Validation metrics')
print(evaluation_summary['validation'])
if 'test_internal' in evaluation_summary:
    print('Internal test metrics')
    print(evaluation_summary['test_internal'])


In [ ]:
# Plot Display
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, filename, title in [
    (axes[0], 'prediction_scatter.png', 'Prediction Scatter'),
    (axes[1], 'residual_plot.png', 'Residual Plot'),
]:
    image = plt.imread(RUN_DIR / filename)
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')
fig.tight_layout()
plt.show()


In [ ]:
# Prediction View
preds = pd.read_csv(RUN_DIR / 'sample_predictions.csv')
display(preds.head(20))

if preds.empty:
    EVAL_VERDICT = 'PATCH_REQUIRED'
else:
    EVAL_VERDICT = 'READY_FOR_NEXT_DECISION'
print('EVAL_VERDICT', EVAL_VERDICT)


## Final Verdict
`READY_FOR_NEXT_DECISION` when validation metrics and plots are produced and sample predictions are inspectable.
